In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display

df_evasao = pd.read_csv("https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/main/Data/02_filtered/Maiores_Taxas_Evasao_e_Reprovacao_2024.csv")
df_censo = pd.read_csv("https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/main/Data/01_Cleaned/Tabela_Censo_Escolar_2024.csv", sep=";")
df_enem = pd.read_csv("https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/main/Data/01_Cleaned/Tabela_ENEM_2024.csv")

# Convertendo a taxa de evasão para numérico e tratando valores ausentes
df_evasao['evasao_medio_total'] = pd.to_numeric(df_evasao['evasao_medio_total'].replace('Não informado', np.nan), errors='coerce')

# Cruzando as tabelas do Censo e da Evasão pela chave da escola
df_escolas_censo = pd.merge(df_censo, df_evasao, left_on='CO_ENTIDADE', right_on='codigo_escola', how='inner')

# Criando a métrica reversa: Taxa de Permanência (Retenção)
df_escolas_censo['taxa_permanencia'] = 100 - df_escolas_censo['evasao_medio_total']

# Agrupando os dados por Unidade Federativa (Estado)
permanencia_uf = df_escolas_censo.groupby('SG_UF')['taxa_permanencia'].mean().reset_index()
enem_uf = df_enem.groupby('SG_UF_PROVA')[['NOTA_GERAL', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']].mean().reset_index()

# Consolidação final do cruzamento por estado (Evasão vs Desempenho)
df_final = pd.merge(permanencia_uf, enem_uf, left_on='SG_UF', right_on='SG_UF_PROVA', how='inner')
df_final = df_final.sort_values(by='taxa_permanencia', ascending=False)

print("--- RANKING DE PERMANÊNCIA NAS ESCOLAS DO CENSO E NOTAS DO ENEM ---")
display(df_final[['SG_UF', 'taxa_permanencia', 'NOTA_GERAL', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']].head(10).round(2))
print("\n")

cor_fundo = '#F8F9FA'
cor_pontos = '#87CEEB'       
cor_tendencia = '#3E4B8E'    
cor_media_nac = '#6E3377'    
cor_texto = '#333333'

fig = px.scatter(
    df_final,
    x='taxa_permanencia',
    y='NOTA_GERAL',
    text='SG_UF',
    trendline='ols',
    title='<b>Taxa de Permanência Escolar vs Nota Média Geral no ENEM</b>',
    labels={
        'taxa_permanencia': 'Taxa de Permanência do Aluno (%)',
        'NOTA_GERAL': 'Nota Média Geral no ENEM'
    }
)

fig.update_traces(
    marker=dict(size=14, color=cor_pontos, line=dict(width=1.5, color='white')),
    textposition='top center',
    textfont=dict(color='#666666', size=11, family="Arial"),
    selector=dict(mode='markers+text')
)

fig.update_traces(
    line=dict(color=cor_tendencia, width=3, dash='solid'),
    selector=dict(mode='lines')
)

media_nacional = df_enem['NOTA_GERAL'].mean()
fig.add_hline(
    y=media_nacional,
    line_dash="dash",
    line_color=cor_media_nac,
    line_width=2,
    annotation_text=f"Média Nacional: {media_nacional:.1f}",
    annotation_position="bottom right",
    annotation_font=dict(color=cor_media_nac, size=12, family="Arial")
)

fig.update_layout(
    plot_bgcolor=cor_fundo,    
    paper_bgcolor=cor_fundo,   
    font=dict(family="Arial, sans-serif", color=cor_texto),
    title_font=dict(size=18, color="#111111"),
    xaxis=dict(showgrid=True, gridcolor='#EBEBEB', zeroline=False),
    yaxis=dict(showgrid=True, gridcolor='#EBEBEB', zeroline=False),
    margin=dict(l=40, r=40, t=70, b=40),
    hovermode="closest"
)

fig.show()